# Solutions: Build the Notepad

**Language:** Python
**Topics:** statelessness, RAIL, LCEL, MessagesPlaceholder, InMemoryChatMessageHistory, RunnableWithMessageHistory, session isolation
**Level:** Intermediate

Every answer is worked in full and run for real. Where a question is about code, you will see the code, its output, and a plain-language walk through the steps. Where a bug is involved, you will watch it break first, then get fixed.

**How the offline model answers** (so the outputs make sense):

| The last human message | LocalAgent replies |
|---|---|
| contains a PNR earlier and ends with a PNR question | `Your PNR is <PNR>.` |
| asks for a PNR but none is in view | `I do not have your PNR. Could you share it?` |
| mentions cancel or leg | `I see the BLR to DEL leg on your booking...` |
| anything else | `Noted: <that message>` |

Run the scaffold once, then read top to bottom.

## Scaffold

In [ ]:
# pip install langchain langchain-core
import re, warnings
warnings.filterwarnings("ignore", message=".*RunnableWithMessageHistory.*")
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

class LocalAgent(BaseChatModel):
    # Offline stand-in. Finds a PNR in the messages it is given and answers from it.
    # No API key. Its job is to make the plumbing visible, not to be clever.
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        seen = " ".join(m.content for m in messages if isinstance(m.content, str))
        mm = re.search(r"PNR\s+([A-Z0-9]{5,8})", seen)
        pnr = mm.group(1) if mm else None
        last = next((m.content for m in reversed(messages)
                     if m.type == "human" and isinstance(m.content, str)), "")
        low = last.lower()
        if "pnr" in low and "?" in last:
            r = f"Your PNR is {pnr}." if pnr else "I do not have your PNR. Could you share it?"
        elif "cancel" in low or "leg" in low:
            r = "I see the BLR to DEL leg on your booking. I can help with that."
        elif last:
            r = f"Noted: {last}"
        else:
            r = "How can I help with your booking today?"
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=r))])
    @property
    def _llm_type(self):
        return "local-agent"

def show(title, messages):
    print(f"[{title}]  {len(messages)} message(s) the model sees:")
    for i, m in enumerate(messages, 1):
        text = m.content if isinstance(m.content, str) else str(m.content)
        print(f"   {i}. {m.type:6} | {text}")

model = LocalAgent()
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise airline support agent."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | model | StrOutputParser()
print("Scaffold ready. Model:", type(model).__name__)

## Q1. Predict the output  ·  Predict

**Answer**
```
Noted: I am Rao, PNR JX48Q2, Gold tier.
I do not have your PNR. Could you share it?
```

**Why, step by step**
1. Call one hands the model a list with one message: the fact.
2. The model answers, then forgets. It keeps nothing.
3. Call two hands the model a brand new list with one message: the question.
4. That list has no PNR in it, so the honest answer is "I do not have it."

```mermaid
flowchart LR
    C1[call 1: I am Rao PNR JX48Q2] --> M1[model] --> R1[Noted...]
    C2[call 2: what is my PNR] --> M2[model] --> R2[I do not have your PNR]
    R1 -. no link between calls .-> C2
```

The two calls are strangers. That is the Goldfish Principle.

In [ ]:
r1 = model.invoke([HumanMessage("I am Rao, PNR JX48Q2, Gold tier.")])
r2 = model.invoke([HumanMessage("What is my PNR?")])
print(r1.content)
print(r2.content)

## Q2. Predict the output  ·  Predict

**Answer**
```
5
Your PNR is JX48Q2.
```

**Why, step by step**
1. Start the notepad with one system message.
2. Turn 1 appends the human line, gets a reply, appends the reply. Now 3 messages.
3. Turn 2 appends the question, gets a reply, appends it. Now 5 messages.
4. On turn 2 the PNR from turn 1 is still on the list, so the model reads it back.

```mermaid
flowchart LR
    U[user turn] --> A[append human] --> I[invoke on full list] --> L[append reply] --> U
```

Watch the notepad grow, printed below.

In [ ]:
messages = [SystemMessage("You are a concise airline support agent.")]

def ask(user_text):
    messages.append(HumanMessage(user_text))
    reply = model.invoke(messages)
    messages.append(reply)
    return reply.content

ask("I am Rao, PNR JX48Q2, Gold tier.")
ask("What is my PNR?")
print(len(messages))
print(messages[-1].content)
print()
show("after 2 turns", messages)

## Q3. Read the code  ·  Read

**Answer: (b)** the pipe feeds each stage's output into the next, left to right.

```mermaid
flowchart LR
    IN[input dict] --> P[prompt] --> M[model] --> S[StrOutputParser] --> OUT[final string]
```

- `prompt` turns your dict into messages.
- `model` turns messages into a chat reply.
- `StrOutputParser` turns that reply into a plain string.

Each block speaks the same `invoke` interface, so they snap together. Proof: the final type is a plain `str`.

In [ ]:
out = chain.invoke({"input": "I am Rao, PNR JX48Q2. What is my PNR?", "history": []})
print("final type:", type(out).__name__)
print("value     :", out)

## Q4. Multi-select  ·  Read

**Answer: real are** `.messages`, `.add_user_message()`, `.add_ai_message()`, `.clear()`, `.add_messages([...])`.
**Fake:** `.append()` and `.get_last()`.

The store is a list with manners: you add turns through named methods, not a raw list `append`. The cell below proves which names exist.

In [ ]:
h = InMemoryChatMessageHistory()
h.add_user_message("I am Rao, PNR JX48Q2.")
h.add_ai_message("Noted.")
h.add_messages([HumanMessage("one more")])

print("stored roles     :", [m.type for m in h.messages])
for name in ["add_user_message", "add_ai_message", "add_messages", "clear", "append", "get_last"]:
    print(f"  has {name:18}: {hasattr(h, name)}")

h.clear()
print("after clear()    :", len(h.messages), "messages")

## Q5. True or false  ·  Read

| # | Statement | Verdict |
|---|---|---|
| 1 | placeholder keeps history role-typed | True |
| 2 | placeholder name must match `history_messages_key` | True |
| 3 | placeholder flattens history into one string | False |
| 4 | placeholder position does not matter | False |

Statement 3 is the trap. Flattening is the exact thing the placeholder prevents. Below: a flattened blob versus role-typed messages. Only the second keeps who said what.

In [ ]:
typed = [HumanMessage("I am Rao."), AIMessage("Noted, Rao.")]

flat = "\n".join(m.content for m in typed)
print("Flattened to one string (what you avoid):")
print("  ", repr(flat))
print()
print("Role-typed (what MessagesPlaceholder keeps):")
for m in typed:
    print("  ", m.type, "->", m.content)

## Q6. Trace the flow  ·  Trace

**Answer:** `system, h1, a1, h2, a2, h3` (six messages).

Append human runs before invoke, so the turn 3 question is on the list. The reply is appended after the call, so it is not in this call yet.

```mermaid
flowchart LR
    Q[turn 3 question] --> AP[append human] --> INV[invoke sees 6 messages] --> AL[append reply] --> NX[turn 4]
```

In [ ]:
messages = [SystemMessage("You are a concise airline support agent."),
            HumanMessage("h1"), AIMessage("a1"),
            HumanMessage("h2"), AIMessage("a2")]

messages.append(HumanMessage("h3"))   # the turn 3 question arrives
show("what invoke receives on turn 3", messages)

## Q7. Match the code to the RAIL step  ·  Match

| Code | RAIL step |
|---|---|
| `get_session_history(session_id)` | Retrieve |
| filling `MessagesPlaceholder("history")` | Augment |
| `model.invoke(messages)` | Invoke |
| `history.add_ai_message(reply)` | Log |

```mermaid
flowchart LR
    R[Retrieve past turns] --> A[Augment the prompt] --> I[Invoke the model] --> L[Log the exchange] --> R
```

Every memory tool, from a six line loop to a checkpointer, is these four steps wearing a costume.

## Q8. Debug: the missing Log  ·  Debug

**The buggy code**
```python
def ask(user_text):
    messages.append(HumanMessage(user_text))
    reply = model.invoke(messages)
    return reply.content           # reply is never appended
```

**Bad line:** none appends the reply. **Fix:** add `messages.append(reply)` before `return`.

First watch it break: the notepad ends with zero ai turns, so the bot cannot remember what it said.

In [ ]:
# BROKEN
messages = [SystemMessage("Airline support.")]
def ask_bug(user_text):
    messages.append(HumanMessage(user_text))
    reply = model.invoke(messages)
    return reply.content            # missing the Log step

ask_bug("I am Rao, PNR JX48Q2.")
ask_bug("What did I just tell you?")
show("BROKEN: no ai turns logged", messages)
print("ai turns on the notepad:", sum(1 for m in messages if m.type == "ai"))

In [ ]:
# FIXED
messages = [SystemMessage("Airline support.")]
def ask_fixed(user_text):
    messages.append(HumanMessage(user_text))
    reply = model.invoke(messages)
    messages.append(reply)          # the fix
    return reply.content

ask_fixed("I am Rao, PNR JX48Q2.")
print(ask_fixed("What is my PNR?"))
show("FIXED: ai turns present", messages)

## Q9. Debug: wrong order  ·  Debug

**The buggy code**
```python
def ask(user_text):
    reply = model.invoke(messages)          # too early
    messages.append(HumanMessage(user_text))
    messages.append(reply)
    return reply.content
```

**Bad line:** `model.invoke(messages)` runs before the new question is added, so Augment sees stale history. **Fix:** append the human line first, then invoke.

Below, the notepad shows the reply sitting before the question it was meant to answer.

In [ ]:
# BROKEN
messages = [SystemMessage("Airline support.")]
def ask_bug(user_text):
    reply = model.invoke(messages)          # invoked before the question exists
    messages.append(HumanMessage(user_text))
    messages.append(reply)
    return reply.content

print("turn 1 reply:", ask_bug("I am Rao, PNR JX48Q2."))
show("BROKEN: reply precedes its question", messages)

In [ ]:
# FIXED
messages = [SystemMessage("Airline support.")]
def ask_fixed(user_text):
    messages.append(HumanMessage(user_text))   # Augment first
    reply = model.invoke(messages)             # then Invoke
    messages.append(reply)                     # then Log
    return reply.content

ask_fixed("I am Rao, PNR JX48Q2.")
print(ask_fixed("What is my PNR?"))
show("FIXED: question, then answer", messages)

## Q10. Debug: the key mismatch  ·  Debug

**The buggy code**
```python
bot = RunnableWithMessageHistory(
    chain, get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",   # prompt slot is named "history"
)
```

**Bad line:** `history_messages_key="chat_history"` does not match `MessagesPlaceholder("history")`. **Fix:** use `history_messages_key="history"`.

This one does not fail quietly. Invoking it raises, because the prompt is left with an unfilled `history` slot. Watch the actual error, then the fix.

In [ ]:
store = {}
def gsh(sid):
    if sid not in store:
        store[sid] = InMemoryChatMessageHistory()
    return store[sid]

# BROKEN: key does not match the placeholder name
bad = RunnableWithMessageHistory(chain, gsh,
    input_messages_key="input", history_messages_key="chat_history")
try:
    bad.invoke({"input": "hi"}, config={"configurable": {"session_id": "x"}})
except Exception as e:
    print("Error you get:", type(e).__name__)
    print(str(e).splitlines()[0])

In [ ]:
# FIXED: key matches MessagesPlaceholder("history")
good = RunnableWithMessageHistory(chain, gsh,
    input_messages_key="input", history_messages_key="history")

good.invoke({"input": "I am Rao, PNR JX48Q2."},
            config={"configurable": {"session_id": "y"}})
print(good.invoke({"input": "What is my PNR?"},
                  config={"configurable": {"session_id": "y"}}))

## Q11. Debug: one shared store  ·  Debug

**The buggy code**
```python
store = InMemoryChatMessageHistory()   # one history for everyone
def get_session_history(session_id):
    return store                       # ignores who is asking
```

**Bad:** every `session_id` gets the same notepad. **Fix:** a dict keyed by `session_id`.

```mermaid
flowchart LR
    RAO[Rao writes PNR] --> ONE[one shared history]
    B[other customer asks] --> ONE
    ONE --> LEAK[other customer is told Rao's PNR]
```

Watch the leak, then the fix that isolates them.

In [ ]:
# BROKEN: shared history, identity ignored
shared = InMemoryChatMessageHistory()
def gsh_bug(sid):
    return shared

bot = RunnableWithMessageHistory(chain, gsh_bug,
    input_messages_key="input", history_messages_key="history")
bot.invoke({"input": "I am Rao, PNR JX48Q2."},
           config={"configurable": {"session_id": "rao"}})
leak = bot.invoke({"input": "What is my PNR?"},
                  config={"configurable": {"session_id": "someone-else"}})
print("A different customer asked, and got:", leak, " <-- LEAK")

In [ ]:
# FIXED: one notepad per session_id
store = {}
def gsh_fixed(sid):
    if sid not in store:
        store[sid] = InMemoryChatMessageHistory()
    return store[sid]

bot = RunnableWithMessageHistory(chain, gsh_fixed,
    input_messages_key="input", history_messages_key="history")
bot.invoke({"input": "I am Rao, PNR JX48Q2."},
           config={"configurable": {"session_id": "rao"}})
clean = bot.invoke({"input": "What is my PNR?"},
                   config={"configurable": {"session_id": "someone-else"}})
print("Different customer now gets:", clean, " <-- isolated")

## Q12. Diagram to code: boilerplate  ·  Build

```mermaid
flowchart LR
    RAO[id cust-rao] --> GS[get_session_history keyed by id]
    OTH[id cust-x] --> GS
    GS --> HR[history cust-rao]
    GS --> HX[history cust-x]
```

**Solution:** a per-session store plus the two matching keys. Steps:
1. Keep a dict from id to history.
2. Create a history the first time an id appears.
3. Set `input_messages_key="input"` and `history_messages_key="history"`.

In [ ]:
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

bot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

bot.invoke({"input": "I am Rao, PNR JX48Q2."}, config={"configurable": {"session_id": "cust-rao"}})
print(bot.invoke({"input": "What is my PNR?"}, config={"configurable": {"session_id": "cust-rao"}}))
print("sessions tracked:", list(store.keys()))

## Q13. Diagram to code: bare  ·  Build

```mermaid
flowchart LR
    IN[input dict] --> P[prompt system, history slot, human input] --> M[model] --> S[StrOutputParser] --> OUT[string]
```

**Solution:** three template lines plus a pipe. The history slot name `history` is what you will hand to `history_messages_key` later.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise airline support agent."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | model | StrOutputParser()

print(chain.invoke({"input": "I am Rao, PNR JX48Q2. What is my PNR?", "history": []}))

## Q14. Predict the output  ·  Predict

**Answer**
```
Your PNR is ZZ90Q1.
2 4
```

**Why, step by step**
1. Turn 1 writes to session `rao`. Turn 2 writes to session `mehta`. Two separate notepads.
2. The PNR question runs on `mehta`, whose notepad only holds Mehta's PNR, so the answer is ZZ90Q1.
3. `rao` had one turn: human + ai = 2 messages. `mehta` had two turns = 4 messages.
4. The system line lives in the prompt template, not in the stored history, so it never counts.

```mermaid
flowchart LR
    R1[rao: I am Rao ...] --> SR[store rao: 2 msgs]
    M1[mehta: I am Mehta ...] --> SM[store mehta]
    M2[mehta: what is my PNR] --> SM
    SM --> A[answer ZZ90Q1, store mehta: 4 msgs]
```

In [ ]:
store = {}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

bot = RunnableWithMessageHistory(chain, get_session_history,
    input_messages_key="input", history_messages_key="history")

bot.invoke({"input": "I am Rao, PNR JX48Q2."},   config={"configurable": {"session_id": "rao"}})
bot.invoke({"input": "I am Mehta, PNR ZZ90Q1."}, config={"configurable": {"session_id": "mehta"}})
ans = bot.invoke({"input": "What is my PNR?"},   config={"configurable": {"session_id": "mehta"}})
print(ans)
print(len(store["rao"].messages), len(store["mehta"].messages))

**Skeptic asks:** if the store never held the system line, how did the model still act like an agent? The prompt template re-adds it on every call, then it is dropped. Re-sent, never remembered. That is the whole session in one sentence.

You have RAIL, LCEL, the primitives, and isolation. Next notebook: keep all of that alive in production.